# Trabalho 1 — Aquisição de Dados (Cinema)

Coleta TMDB (API) + OMDb (API) + Letterboxd (scraping), integração e limpeza.

**Antes de rodar:** preencha `TMDB_API_KEY` e `OMDB_API_KEY` em `.env` (veja o README).

## 0. Setup e utilitários

In [ ]:
from __future__ import annotations

import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from tqdm.auto import tqdm

ROOT = Path.cwd()
if not (ROOT / "requirements.txt").exists() and (ROOT / "trabalho01" / "requirements.txt").exists():
    ROOT = ROOT / "trabalho01"

DIR_BRUTOS = ROOT / "dados_brutos"
DIR_TRATADOS = ROOT / "dados_tratados"
DIR_DOCS = ROOT / "docs"
LOG_PATH = DIR_DOCS / "proveniencia.jsonl"

for d in (DIR_BRUTOS, DIR_TRATADOS, DIR_DOCS):
    d.mkdir(parents=True, exist_ok=True)

load_dotenv(ROOT / ".env")
TMDB_API_KEY = os.getenv("TMDB_API_KEY", "").strip()
OMDB_API_KEY = os.getenv("OMDB_API_KEY", "").strip()

HEADERS = {
    "User-Agent": (
        "UFAM-CD-Trabalho1-Cinema/1.0 "
        "(+academic; contato via repositorio do projeto)"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

print("ROOT:", ROOT)
print("TMDB_API_KEY definida:", bool(TMDB_API_KEY))
print("OMDB_API_KEY definida:", bool(OMDB_API_KEY))
if not TMDB_API_KEY or not OMDB_API_KEY:
    print("AVISO: preencha trabalho01/.env com TMDB_API_KEY e OMDB_API_KEY antes da coleta.")

In [ ]:
def log_proveniencia(
    fonte: str,
    url: str,
    metodo: str,
    status: int | None,
    params: dict | None = None,
    observacao: str = "",
) -> None:
    """Append de uma linha JSON no registro de proveniência."""
    registro = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "fonte": fonte,
        "url": url,
        "metodo": metodo,
        "params": params or {},
        "status": status,
        "observacao": observacao,
    }
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(registro, ensure_ascii=False) + "\n")


def get_com_retry(
    url: str,
    *,
    params: dict | None = None,
    headers: dict | None = None,
    timeout: int = 30,
    max_tentativas: int = 3,
    sleep_base: float = 1.0,
) -> requests.Response:
    """GET com retentativas para 429/5xx."""
    ultimo_erro: Exception | None = None
    for tentativa in range(1, max_tentativas + 1):
        try:
            resp = requests.get(
                url,
                params=params,
                headers=headers or HEADERS,
                timeout=timeout,
            )
            if resp.status_code in {429, 500, 502, 503, 504}:
                time.sleep(sleep_base * tentativa)
                continue
            return resp
        except requests.RequestException as exc:
            ultimo_erro = exc
            time.sleep(sleep_base * tentativa)
    if ultimo_erro:
        raise ultimo_erro
    raise RuntimeError(f"Falha ao obter {url}")


print("Utilitários carregados. Log:", LOG_PATH)

## 1. Teste das API keys (rode após preencher o `.env`)

In [ ]:
if TMDB_API_KEY:
    r_tmdb = get_com_retry(
        "https://api.themoviedb.org/3/movie/550",
        params={"api_key": TMDB_API_KEY},
    )
    print("TMDB:", r_tmdb.status_code, r_tmdb.json().get("title"))
    log_proveniencia(
        "TMDB",
        r_tmdb.url.replace(TMDB_API_KEY, "***"),
        "API GET /movie/550",
        r_tmdb.status_code,
        {"movie_id": 550},
        "teste de chave",
    )
else:
    print("TMDB: chave ausente")

if OMDB_API_KEY:
    r_omdb = get_com_retry(
        "https://www.omdbapi.com/",
        params={"i": "tt0137523", "apikey": OMDB_API_KEY},
    )
    data = r_omdb.json()
    print("OMDb:", r_omdb.status_code, data.get("Title"), data.get("imdbRating"), data.get("Metascore"))
    log_proveniencia(
        "OMDb",
        "https://www.omdbapi.com/?i=tt0137523&apikey=***",
        "API GET",
        r_omdb.status_code,
        {"i": "tt0137523"},
        "teste de chave",
    )
else:
    print("OMDb: chave ausente")

## Próximos passos (ainda não implementados neste passo)

1. Coleta TMDB → `dados_brutos/tmdb_raw.csv`
2. Coleta OMDb → `dados_brutos/omdb_raw.csv`
3. Scraping Letterboxd → `dados_brutos/letterboxd_raw.csv`
4. Join + limpeza → `dados_tratados/base_tratada.parquet`
5. Dataset Card + proveniência completa